In [ ]:
import nltk
from nltk.corpus import wordnet
from nltk.wsd import lesk
from nltk.tokenize import word_tokenize

In [ ]:
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('punkt_tab', quiet=True)

## Extract WordNet Semantic Relations for a Synset

In [ ]:
def get_related_words(synset):
    related = set()

    related.update(
        [lemma.name().replace('_', ' ') for lemma in synset.lemmas()]
    )

    for relation in [
        synset.hypernyms(),
        synset.hyponyms(),
        synset.member_meronyms(),
        synset.substance_meronyms(),
        synset.part_meronyms(),
        synset.similar_tos(),
        synset.also_sees(),
        synset.entailments(),
    ]:
        for s in relation:
            related.update(
                [lemma.name().replace('_', ' ') for lemma in s.lemmas()]
            )

    return related

## Walker's Algorithm — Sense Scoring via Semantic Relations

In [ ]:
def walker_wsd(context_sentence, target_word):
    tokens = word_tokenize(context_sentence)
    context_words = set(
        w.lower()
        for w in tokens
        if w.isalpha() and len(w) > 2 and w.lower() != target_word.lower()
    )

    candidate_synsets = wordnet.synsets(target_word)

    best_synset = None
    best_score = -1
    scores = {}

    for synset in candidate_synsets:
        related = get_related_words(synset)
        related_lower = set(w.lower() for w in related)
        overlap = context_words & related_lower
        score = len(overlap)

        scores[synset.name()] = {
            "score": score,
            "overlap": overlap,
            "definition": synset.definition()
        }

        if score > best_score:
            best_score = score
            best_synset = synset

    return best_synset, scores

## Lesk Algorithm (for comparison)

In [ ]:
def lesk_wsd(context_sentence, target_word):
    tokens = word_tokenize(context_sentence)
    synset = lesk(tokens, target_word)
    if synset is None:
        return "UNKNOWN", "No sense found"
    return synset.name(), synset.definition()

## Test on Ambiguous Words

In [ ]:
test_cases = [
    ("The bank deposited the funds into her account.", "bank"),
    ("We sat on the river bank and watched the sunset.", "bank"),
    ("The baseball player swung the bat.", "bat"),
    ("The bat flew out of the cave at dusk.", "bat"),
    ("She is a star in the movie industry.", "star"),
    ("The sun is the brightest star in our sky.", "star"),
    ("The factory plant employs over 500 workers.", "plant"),
    ("The plant needs more sunlight to grow.", "plant"),
]

## Run Walker's Algorithm and Compare with Lesk

In [ ]:
for sentence, target in test_cases:
    print(f"Sentence: {sentence} | Target: {target}")
    print("=" * 80)

    best_synset, scores = walker_wsd(sentence, target)
    best_name, best_def = (
        best_synset.name(), best_synset.definition()
    ) if best_synset else ("UNKNOWN", "No sense found")

    lesk_name, lesk_def = lesk_wsd(sentence, target)

    print(f"Walker : {best_name} | Score: {scores.get(best_name, {}).get('score', 0)} | Overlap: {scores.get(best_name, {}).get('overlap', set())}")
    print(f"Walker : {best_def}")
    print(f"Lesk  : {lesk_name}")
    print(f"Lesk  : {lesk_def}")

    match = "(SAME)" if best_name == lesk_name else "(DIFFERENT)"
    print(f"Match : {match}")
    print("\n")